<a href="https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/ColabFold2_preview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ColabFold2 preview

Predict protein, RNA, DNA and small-molecule structures with [AlphaFold 3](https://www.nature.com/articles/s41586-024-07487-w), running **any** of fourteen models through one implementation. Pick a model in the install cell — the weights download themselves.

| model | weights from | licence |
|---|---|---|
| `openbind0` | [OpenBind0 / OpenFold3 v0.5.0](https://github.com/aqlaboratory/openfold-3/releases/tag/v0.5.0) (AlQuraishi Lab) | Apache-2.0 |
| `openfold3` | [OpenFold3 preview-2](https://github.com/aqlaboratory/openfold) (AlQuraishi Lab) | Apache-2.0 |
| `boltz2` | [Boltz-2](https://github.com/jwohlwend/boltz) (MIT / Jeremy Wohlwend et al.) | MIT |
| `protenix2` | [Protenix-v2](https://github.com/bytedance/Protenix) (ByteDance) | Apache-2.0 |
| `rosettafold3` | [RoseTTAFold3](https://github.com/RosettaCommons/foundry) (RosettaCommons) | BSD-3-Clause |
| `chai1` | [chai-1](https://github.com/chaidiscovery/chai-lab) (Chai Discovery) | Apache-2.0 |
| `intellifold2` | [IntelliFold-v2](https://huggingface.co/intelligenAI/intellifold) (IntelligenAI) | Apache-2.0 |
| `opendde` | [OpenDDE](https://huggingface.co/aurekaresearch/OpenDDE) (Aureka Research) | Apache-2.0 |
| `esmfold2` | [ESMFold2](https://huggingface.co/biohub/ESMFold2) (Arc Institute / Biohub) | MIT |
| `esmfold2_lm600m` | ESMFold2 against the 600M ESM-C tower | MIT |
| `esmfold2_lm300m` | ESMFold2 against the 300M ESM-C tower | MIT |
| `af2_ptm` | AlphaFold 2 monomer pTM (DeepMind) | CC BY 4.0 |
| `af2_multimer` | AlphaFold 2 multimer v3 (DeepMind) | CC BY 4.0 |
| `alphafold3` | Google DeepMind's own parameters | [AF3 terms of use](https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md) |

Twelve of them run through the **same** JAX/Haiku AlphaFold 3 graph — only the weights and a few gated forward branches differ — so the input box, the MSA path, the outputs, the confidence metrics and the plots are identical whichever you pick. Switch models by changing one dropdown and re-running.

MSA generation via the [ColabFold](https://github.com/sokrypton/ColabFold) MMseqs2 server — **no local databases required**. Attention/XLA flags are chosen automatically for your runtime (T4, L4/Ada, A100/H100, or CPU).

**Citations:**
- Abramson et al. (2024) AlphaFold 3. *Nature* [doi:10.1038/s41586-024-07487-w](https://doi.org/10.1038/s41586-024-07487-w)
- Mirdita et al. (2022) ColabFold. *Nature Methods* [doi:10.1038/s41592-022-01488-1](https://doi.org/10.1038/s41592-022-01488-1)
- Whichever model you run — please cite it too; each links to its source above.

**Credits:** AF3 code: Google DeepMind (Apache 2.0) · weights: each model's authors, as listed.


In [ ]:
#@title Install dependencies (~3 mins)
%%time
import os, time, glob, shutil, sys

model = "openbind0" #@param ["openbind0", "openfold3", "boltz2", "protenix2", "rosettafold3", "chai1", "intellifold2", "opendde", "esmfold2", "esmfold2_lm600m", "esmfold2_lm300m", "alphafold3", "af2_ptm", "af2_multimer"]
#@markdown - **model**: which set of weights to run. All of them use the same AlphaFold 3
#@markdown   graph, so everything downstream is identical. `openbind0` is OpenFold3's current
#@markdown   release and a good default; `openfold3` is their earlier preview-2, kept because
#@markdown   earlier results used it. The three `esmfold2*` entries fold from ESM-C instead
#@markdown   of an MSA -- single sequence, no search -- and differ only in the size of that
#@markdown   language model (6B, 600M, 300M). `chai1` and `esmfold2*` download and run their
#@markdown   language model automatically. `alphafold3` fetches Google DeepMind's own
#@markdown   parameters and is subject to the AF3 terms of use.

persist_cache_to_drive = False #@param {type:"boolean"}
#@markdown - **persist_cache_to_drive**: keep the compiled model in your Google Drive so
#@markdown   the next session does not recompile. Measured on a 68-residue input: the first
#@markdown   prediction takes **69 s** with a cold cache and **16 s** with a warm one, so this
#@markdown   is worth about **53 s per session** (more for longer inputs). Colab wipes `/tmp`
#@markdown   between sessions, which is why it has to go somewhere else to survive. Leaving
#@markdown   it off costs only that recompile; it never changes a result.

# PINNED, both halves. Until 2026-09-16 this installed the v3.1.5 wheel for its
# compiled extension and then overlaid the Python half from the BRANCH HEAD --
# so the notebook mixed a fixed binary with a moving source tree, and two runs
# on different days could be different code. 3.1.7 is published on PyPI
# (`alphafold3-colabfold`, cp312/cp313/cp314 manylinux + macOS arm64) and its
# Python half already knows every model, so the overlay is gone and both the
# package and run_alphafold.py come from one tag.
VERSION = '3.1.7'
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'
IS_AF3 = (model == 'alphafold3')
# AlphaFold 2 is a SIBLING NETWORK, not one of the AF3-family ports: MSA row and
# column attention into an IPA head, reached through the same CLI and writing the
# same outputs. Its parameters are DeepMind's own release under CC BY 4.0, so they
# are fetched from source. Protein only -- a ligand or nucleotide in the input
# raises rather than folding the protein part and saying nothing.
IS_AF2 = model.startswith('af2_')
AF2_DIR = 'af2_params'
# int8 everywhere: same weights stored 8-bit and expanded on load, which is
# what keeps a Colab download to a few hundred MB. Not a knob -- there is no
# reason to pick anything else here, and AlphaFold 3's own parameters come
# from Google as float32 regardless.
PRECISION = 'fp32' if (IS_AF3 or IS_AF2) else 'int8'

if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  os.system("pip install -q 'jax[cuda12]==0.10.1' dm-haiku==0.0.17 rdkit==2025.9.4 \
  zstandard awscli tokamax==0.0.11 py3Dmol py2Dmol")
  # THE FAT WHEEL, from the GitHub release -- not the slim one on PyPI.
  # `alphafold3.cpp` needs libcifpp's components.cif (518 MB raw, 120 MB
  # zipped) and cannot import without it:
  #     ImportError: Could not find the libcifpp components.cif file.
  # With the data the wheel is 130 MB, over PyPI's 100 MB per-file limit, so
  # PyPI carries the slim build (correct for anyone who provisions the data
  # themselves) and the release carries `+data`, which is self-contained.
  _whl = (f'https://github.com/sokrypton/alphafold3/releases/download/v{VERSION}'
          f'/alphafold3_colabfold-{VERSION}%2Bdata-cp313-cp313'
          f'-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl')
  os.system(f"pip install -q --no-deps '{_whl}'")
  # `run_alphafold.py` is a top-level script, not part of the package
  # (`wheel.packages = ["src/alphafold3"]`), so the wheel does not carry it.
  # Fetch it AT THE TAG so the driver and the library are the same commit.
  os.system(f'wget -q -O run_alphafold.py https://raw.githubusercontent.com'
            f'/sokrypton/alphafold3/v{VERSION}/run_alphafold.py')
  # haiku 0.0.17 still calls the moved `jax.core.DropVar`; checked against the
  # installed 0.0.17 tree, this one is still needed. (A second sed for
  # `jax.core.get_opaque_trace_state` used to sit here and never matched --
  # base.py reaches it through a `jax_core` alias and already falls back to
  # `jex_core` itself, so it was only ever a no-op.)
  os.system("sed -i 's/jax.core.DropVar/jax.extend.core.DropVar/g' /usr/local/lib/python*/dist-packages/haiku/_src/jaxpr_info.py")
  os.system('touch ALPHAFOLD3_READY')
  print('Packages installed.')

# Patch tokamax so Ada/consumer GPUs (L4, A10, RTX 30/40; cc 8.6/8.9) fall back to XLA
# kernels. tokamax enables its Triton kernels for ALL cc>=8.0 GPUs, but those kernels
# need more shared memory than Ada cards have -> 'Shared memory size limit exceeded' at
# launch (which its trace-time fallback can't catch). Restrict Triton to true datacenter
# GPUs (A100 cc 8.0, H100 cc 9.0+); everything else uses XLA, exactly like the T4 path.
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 (8.6/8.9) lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights, in the background. The ported models are fetched by the same code the run
# uses (alphafold3.model.weights.ensure_weights), so the run finds them already there
# and the cache layout cannot drift between the two. AlphaFold 3's own parameters are
# not ours to redistribute, so those come straight from Google.
STAMP = f'WEIGHTS_DONE_{model}_{PRECISION}'
if not os.path.isfile(STAMP):
  if IS_AF2:
    print('Downloading official AlphaFold 2 parameters (CC BY 4.0)...')
    with open('prefetch_af2.py', 'w') as fh:
      fh.write('import sys\n'
               'from alphafold3.model import weights\n'
               'print(weights.ensure_af2_params(sys.argv[1]))\n')
    os.system(f'(python prefetch_af2.py {AF2_DIR} && touch {STAMP}) &')
  elif IS_AF3:
    print("Downloading official AlphaFold 3 weights (public, no login required)...")
    os.makedirs(NATIVE_DIR, exist_ok=True)
    for _f in glob.glob(f'{NATIVE_DIR}/*'):       # keep exactly one model file in the dir
      os.remove(_f)
    os.system(f'(wget -q -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}" && touch {STAMP}) &')
  else:
    print(f'Downloading {model} weights...')
    with open('prefetch_weights.py', 'w') as fh:
      fh.write('import sys\n'
               'from alphafold3.model import weights\n'
               'print(weights.ensure_weights(sys.argv[1], None, precision=sys.argv[2]))\n')
    os.system(f'(python prefetch_weights.py {model} {PRECISION} && touch {STAMP}) &')

# Where the compiled model is cached. /tmp is wiped when the VM goes away, so a
# fresh session recompiles (~53 s on a small input); Drive survives. Opt-in, and
# the run falls back to /tmp if the mount does not work rather than failing.
CACHE_DIR = '/tmp/af3_cache'
if persist_cache_to_drive:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/.af3_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Compile cache: {CACHE_DIR} (survives this session)')
  except Exception as _e:
    print(f'(Drive mount failed, using {CACHE_DIR}: {_e})')

# Build AF3 data files (background, independent of weights)
if not os.path.isfile('DATA_DONE'):
  print('Building AF3 data files...')
  os.system('(build_data; touch DATA_DONE) &')

for sentinel in (STAMP, 'DATA_DONE'):
  while not os.path.isfile(sentinel):
    time.sleep(5)
  print(f'{sentinel} ✓')

if IS_AF3 and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('AlphaFold 3 weights download failed or incomplete - re-run this cell.')

print(f'Setup complete!  Model: {model}.')
if model == 'chai1':
  print('NOTE: chai-1 is running WITHOUT ESM2 embeddings, which are most of its token\n'
        '      features. Expect worse structures than chai-lab itself produces.')


In [ ]:
#@title Input sequences
import re, os, json, hashlib

#@markdown ### Molecules
#@markdown Separate multiple chains within a box using `:` (extra colons are fine: `A::::B` == `A:B`). Leave a box empty if unused; full details in the Instructions cell.
protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK' #@param {type:"string"}
dna = '' #@param {type:"string"}
rna = '' #@param {type:"string"}
ligand_ccd = '' #@param {type:"string"}
ligand_smiles = '' #@param {type:"string"}

#@markdown ### Run settings
jobname = 'test' #@param {type:"string"}
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
seeds = '1' #@param {type:"string"}
on_existing = "overwrite" #@param ["overwrite", "skip"]
#@markdown - `msa_mode`: `single_sequence` skips the MSA (faster, lower accuracy).
#@markdown - `seeds`: comma-separated, e.g. `1,2,3`.
#@markdown - `on_existing`: `overwrite` replaces this job's previous results; `skip` keeps them.

# Split a box into entries: collapse colon runs, drop whitespace, skip empties
def split_entries(s):
  s = re.sub(r':+', ':', s).strip(':')
  return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e]

prot_seqs   = [e.upper() for e in split_entries(protein)]
dna_seqs    = [e.upper() for e in split_entries(dna)]
rna_seqs    = [e.upper() for e in split_entries(rna)]
ccd_codes   = [e.upper() for e in split_entries(ligand_ccd)]
smiles_strs = split_entries(ligand_smiles)         # case-sensitive: leave as typed

# Build AF3 chain entities (IDs A, B, C, ... in canonical order)
CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
chains, prot_groups, idx = [], {}, 0

for seq in prot_seqs:
  cid = CHAIN_IDS[idx]; idx += 1
  if seq in prot_groups:                           # merge identical seqs -> homo-oligomer
    ent = prot_groups[seq]
    ids = ent['id'] if isinstance(ent['id'], list) else [ent['id']]
    ent['id'] = ids + [cid]
  else:
    ent = {'id': cid, 'sequence': seq, 'templates': []}
    if msa_mode == 'single_sequence':
      ent.update({'unpairedMsa': f'>query\n{seq}\n', 'pairedMsa': ''})
    prot_groups[seq] = ent
    chains.append({'protein': ent})

for seq in rna_seqs:
  c = {'id': CHAIN_IDS[idx], 'sequence': seq}
  if msa_mode == 'single_sequence':
    c['unpairedMsa'] = f'>query\n{seq}\n'
  chains.append({'rna': c}); idx += 1

for seq in dna_seqs:
  chains.append({'dna': {'id': CHAIN_IDS[idx], 'sequence': seq}}); idx += 1

for code in ccd_codes:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'ccdCodes': [code]}}); idx += 1

for smiles in smiles_strs:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'smiles': smiles}}); idx += 1

if not chains:
  raise ValueError('No valid input found - fill in at least one box.')

# Seeds: pull out integers regardless of separators, dedupe, default to [1]
seed_list = []
for tok in re.findall(r'\d+', seeds):
  v = int(tok)
  if v not in seed_list:
    seed_list.append(v)
if not seed_list:
  seed_list = [1]

# Deterministic, lower-cased job name from inputs+seeds.
# Same input+seeds -> same folder (so re-runs reuse it instead of piling up).
# Lower-cased to match run_alphafold.py's sanitised_name() output directory.
def _flat(mol):
  if 'sequence' in mol: return mol['sequence']
  if 'ccdCodes' in mol: return ','.join(mol['ccdCodes'])
  return mol.get('smiles', '?')
flat = ':'.join(_flat(list(c.values())[0]) for c in chains) + '|seeds=' + ','.join(map(str, seed_list))
basejob = (re.sub(r'\W+', '', ''.join(jobname.split())) or 'job').lower()
jobname = basejob + '_' + hashlib.sha1(flat.encode()).hexdigest()[:5]

# Input JSON goes to a temp dir; ALL results land in ONE folder: af3_output/<jobname>/
INPUT_DIR  = '/tmp/af3_inputs'
OUTPUT_DIR = 'af3_output'
job_dir    = f'{OUTPUT_DIR}/{jobname}'

fold_input = {
    'name': jobname,
    'sequences': chains,
    'modelSeeds': seed_list,
    'dialect': 'alphafold3',
    'version': 1,
}
os.makedirs(INPUT_DIR, exist_ok=True)
json_path = f'{INPUT_DIR}/{jobname}.json'
with open(json_path, 'w') as f:
  json.dump(fold_input, f, indent=2)

print(f'Job "{jobname}"  ->  results will be written to {job_dir}/')
fold_input


In [ ]:
#@title Run the model
%%time
import os, shutil, subprocess, glob

#@markdown Inference settings (defaults match AlphaFold 3 - increase only if needed):
num_recycles = 10 #@param {type:"integer"}
num_diffusion_samples = 5 #@param {type:"integer"}
#@markdown - `num_recycles`: refinement passes through the network (default 10). More can help large/hard targets, but is slower.
#@markdown - `num_diffusion_samples`: candidate structures generated per seed (default 5). Total models = seeds x samples.

num_recycles = max(1, int(num_recycles))
num_diffusion_samples = max(1, int(num_diffusion_samples))

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Re-run policy (one folder per job, no timestamped duplicates):
#   overwrite -> wipe this job's folder and recompute
#   skip      -> if a finished result (.cif) is already there, don't recompute
have_results = os.path.isdir(job_dir) and any(f.endswith('.cif') for f in os.listdir(job_dir))
run_it = not (on_existing == 'skip' and have_results)
if run_it:
  shutil.rmtree(job_dir, ignore_errors=True)   # start clean so exactly one folder is produced

# Pick attention impl + XLA flags from the actual device.
# Triton/cuDNN flash attention need Ampere (compute capability >= 8.0);
# 7.x GPUs (T4=7.5, V100=7.0) and CPU use the portable XLA path, and 7.x
# additionally needs the XLA flag that disables the custom-kernel fusion pass.
def detect_device():
  try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15)
    caps = [float(x) for x in out.stdout.split() if x.strip()]
    if caps:
      return 'gpu', min(caps)
  except Exception:
    pass
  return 'cpu', None

device, cap = detect_device()
nojit = False
xla_flags = []   # extra XLA flags to export for this device (per AlphaFold 3's guidance)

if device == 'cpu':
  flash_impl = 'xla'
  nojit = True
  print('No GPU detected - running on CPU with XLA attention + --nojit (slow, but avoids the compile).')
elif cap < 8.0:
  # T4 / V100 (cc 7.x): XLA attention; disable the custom-kernel fusion pass.
  # (Triton GEMM is not supported on these cards, so it is not disabled here.)
  flash_impl = 'xla'
  xla_flags = ['--xla_disable_hlo_passes=custom-kernel-fusion-rewriter']
  print(f'Pre-Ampere GPU (compute capability {cap}) - XLA attention + custom-kernel fusion disabled.')
elif 8.0 < cap < 9.0:
  # L4 / Ada / consumer Ampere (cc 8.6 / 8.9): limited shared memory. XLA's Triton GEMM
  # kernels exceed it ('Shared memory size limit exceeded'), so disable Triton GEMM
  # (falls back to cuBLAS) and use XLA attention to also avoid the Triton attention kernel.
  flash_impl = 'xla'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Ada/consumer GPU (compute capability {cap}) - XLA attention + Triton GEMM disabled (shared-memory limit).')
else:
  # A100 (cc 8.0) and H100 (cc 9.0+): ample shared memory. Triton flash attention,
  # with Triton GEMM disabled per AlphaFold 3's recommended XLA_FLAGS.
  flash_impl = 'triton'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Datacenter GPU (compute capability {cap}) - Triton flash attention + Triton GEMM disabled.')

# Export XLA flags so the child shell (and JAX inside it) inherit them.
cur = os.environ.get('XLA_FLAGS', '')
for f in xla_flags:
  if f not in cur:
    cur = (cur + ' ' + f).strip()
if cur:
  os.environ['XLA_FLAGS'] = cur

print('XLA_FLAGS =', os.environ.get('XLA_FLAGS', '(unset)'))

# Weights. Every ported model resolves its own cache directory (populated by the
# install cell), so --model_dir is passed only for the two whose parameters come
# from DeepMind directly: AlphaFold 3's, and AlphaFold 2's (CC BY 4.0, fetched
# into af2_params by the install cell).
print(f'Model: {model}')

cmd = [
    'python', 'run_alphafold.py',
    f'--json_path={json_path}',
    f'--model={model}',
    '--norun_data_pipeline',
    f'--output_dir={OUTPUT_DIR}',
    f'--cache_dir={CACHE_DIR}',
    '--force_output_dir',          # reuse af3_output/<jobname>/ instead of a timestamped copy
    f'--flash_attention_implementation={flash_impl}',
    f'--num_recycles={num_recycles}',
    f'--num_diffusion_samples={num_diffusion_samples}',
]
if msa_mode == 'mmseqs2_server':
  cmd.append('--use_msa_server')
# chai-1 folds from ESM2 and ESMFold2 from ESM-C; without it they are a
# different model, not a slightly worse one (a natural protein goes to 5.70 A
# where chai-1 reaches 0.642, and an ESMFold2 variant with no MSA encoder has
# nothing left to fold from at all). Both towers run in-process and download on
# demand, which is why run_alphafold makes it opt-in and this passes it.
if model == 'chai1' or model.startswith('esmfold2'):
  cmd.append('--use_esm_embeddings')
if nojit:
  cmd.append('--nojit')
if IS_AF3 or IS_AF2:
  cmd.append(f'--model_dir={AF2_DIR if IS_AF2 else NATIVE_DIR}')

cmd = ' '.join(cmd)
if run_it:
  print(cmd)
  !{cmd}
  print(f'\nDone -> {job_dir}/')
else:
  print(f'Skipping: results already exist in {job_dir}/  (set on_existing=overwrite to recompute).')


In [ ]:
#@title Display structures + PAE (py2Dmol)
import csv, glob, os, json
import numpy as np
import py2Dmol

load_as_frames = True #@param {type:"boolean"}
viewer_size = 400
#@markdown All predicted models load together, best first (**rank_1, rank_2, ...**), each with its own PAE.
#@markdown - `load_as_frames` **off** -> pick a model from the dropdown.
#@markdown - `load_as_frames` **on**  -> models become frames you can play through (press play / drag the slider).
#@markdown - The interactive PAE matrix sits beside the structure; click or drag-box on it to highlight residues.

# All models in rank order (best first): from the ranking CSV, fall back to globbing.
def collect_models():
  ranking_csv = f'{job_dir}/{jobname}_ranking_scores.csv'
  cifs = []
  if os.path.exists(ranking_csv):
    rows = []
    with open(ranking_csv) as f:
      for r in csv.DictReader(f):
        rows.append((float(r['ranking_score']), int(r['seed']), int(r['sample'])))
    for _, seed, sample in sorted(rows, reverse=True):
      d = f'{job_dir}/seed-{seed}_sample-{sample}'
      hit = sorted(glob.glob(f'{d}/*_model.cif')) or sorted(glob.glob(f'{d}/*.cif'))
      if hit:
        cifs.append(hit[0])
  if not cifs:
    cifs = (sorted(glob.glob(f'{job_dir}/**/*_model.cif', recursive=True))
            or sorted(glob.glob(f'{job_dir}/**/*.cif', recursive=True)))
  return cifs

# Per-model PAE: confidences.json next to the CIF, else the top-level one.
def load_pae(cif):
  d = os.path.dirname(cif)
  cands = [p for p in glob.glob(f'{d}/*_confidences.json')
           if 'summary' not in os.path.basename(p)]
  if not cands:
    top = f'{job_dir}/{jobname}_confidences.json'
    cands = [top] if os.path.exists(top) else []
  if cands:
    pae = json.load(open(cands[0])).get('pae')
    if pae is not None:
      return np.asarray(pae, dtype=float)
  return None

cifs = collect_models()
if not cifs:
  raise FileNotFoundError(f'No model CIFs found in {job_dir}/')
print(f'Loaded {len(cifs)} model(s) from {job_dir}/'
      + ('  (frames - press play)' if load_as_frames else '  (use the dropdown to switch)'))

viewer = py2Dmol.view(size=(viewer_size, viewer_size),
                      pae=True, autoplay=load_as_frames)
for i, cif in enumerate(cifs, start=1):
  pae = load_pae(cif)
  if load_as_frames:
    viewer.add_pdb(cif, name='models', paes=pae)      # same name -> frames (play through)
  else:
    viewer.add_pdb(cif, name=f'rank_{i}', paes=pae)    # distinct names -> dropdown of objects
viewer.show()


In [ ]:
#@title Quality metrics and plots
import json, os
import numpy as np
import matplotlib.pyplot as plt

conf_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_confidences.json'
summ_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_summary_confidences.json'

with open(conf_path) as f:
  conf = json.load(f)
with open(summ_path) as f:
  summ = json.load(f)

plddts = np.array(conf.get('atom_plddts', conf.get('token_plddts', [])), dtype=float)
plddt_chain_ids = conf.get('atom_chain_ids', conf.get('token_chain_ids', []))   # pLDDT is per-ATOM
token_chain_ids = conf.get('token_chain_ids', [])                                # PAE is per-TOKEN
pae = np.array(conf.get('pae', []), dtype=float)

# ── Summary (ipTM is None for single-chain jobs — guard before formatting) ─
def fmt(v):
  return f'{v:.3f}' if isinstance(v, (int, float)) else 'n/a'

mean_plddt = summ.get('mean_plddt')
if mean_plddt is None and plddts.size:
  mean_plddt = float(np.mean(plddts))
iptm = summ.get('iptm')

print('=' * 38)
print(f'Mean pLDDT     : {fmt(mean_plddt)}')
print(f'pTM            : {fmt(summ.get("ptm"))}')
print(f'ipTM           : {fmt(iptm)}' + ('   (single chain — no interface)' if iptm is None else ''))
print(f'Ranking score  : {fmt(summ.get("ranking_score"))}')
print('=' * 38)

# ── Plots ───────────────────────────────────────────────────
has_pae = pae.ndim == 2 and pae.size > 0
ncols = 2 if has_pae else 1
fig, axes = plt.subplots(1, ncols, figsize=(13 if has_pae else 6.5, 4))
axes = np.atleast_1d(axes)

# pLDDT per residue — a line (coloured per chain when there is more than one)
ax = axes[0]
x = np.arange(len(plddts))
xmax = max(len(plddts) - 1, 1)
ax.set_xlim(0, xmax)
ax.set_ylim(0, 100)

unique_chains = list(dict.fromkeys(plddt_chain_ids))
if len(unique_chains) > 1:
  colors = plt.cm.tab10(np.linspace(0, 1, len(unique_chains)))
  tcid = np.array(plddt_chain_ids)
  for ch, col in zip(unique_chains, colors):
    y = np.where(tcid == ch, plddts, np.nan)   # NaN gaps keep chains as separate lines
    ax.plot(x, y, lw=1.5, color=col, label=f'Chain {ch}')
  for b in [i for i in range(1, len(plddt_chain_ids)) if plddt_chain_ids[i] != plddt_chain_ids[i-1]]:
    ax.axvline(b - 0.5, color='grey', lw=0.6, alpha=0.5)
  ax.legend(loc='lower right', fontsize=8)
else:
  ax.plot(x, plddts, lw=1.5, color='#1f77b4')

for y in (50, 70, 90):
  ax.axhline(y, ls='--', lw=0.7, color='grey', alpha=0.5)
  ax.text(xmax, y, f' {y}', va='center', ha='left', fontsize=7, color='grey')
ax.set_xlabel('Atom')
ax.set_ylabel('pLDDT')
ax.set_title('Predicted pLDDT per atom')

# PAE matrix
if has_pae:
  ax = axes[1]
  im = ax.imshow(pae, cmap='bwr', vmin=0, vmax=30, interpolation='nearest')
  plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='PAE (Å)')
  if token_chain_ids:
    for b in [i for i in range(1, len(token_chain_ids)) if token_chain_ids[i] != token_chain_ids[i-1]]:
      ax.axhline(b - 0.5, c='black', lw=0.8)
      ax.axvline(b - 0.5, c='black', lw=0.8)
  ax.set_xlabel('Scored residue')
  ax.set_ylabel('Aligned residue')
  ax.set_title('Predicted Aligned Error (PAE)')

plt.tight_layout()
plt.show()


In [ ]:
#@title Download results
from google.colab import files
import os

results_zip = f'{jobname}.result.zip'
os.system(f'zip -r {results_zip} {OUTPUT_DIR}/{jobname}')
files.download(results_zip)


# Instructions <a name="Instructions"></a>

**Quick start**
1. Pick a **model** in the install cell.
2. Fill in the sequence(s) in the **Input sequence(s)** cell.
3. Press **Runtime → Run all**.
4. The install cell (first run only) downloads the weights for the model you picked and builds AF3 data files in the background — subsequent runs reuse them.

---

## Choosing a model

Pick one in the **model** dropdown of the install cell. Twelve of the fourteen are
the same AlphaFold 3 network with different trained weights, so nothing else in the
notebook changes — same input boxes, same MSA path, same outputs and plots. The two
`af2_*` entries are AlphaFold 2, a different network reached through the same CLI.

| model | notes |
|---|---|
| `openbind0` | OpenFold3 v0.5.0 "OpenBind", Apache-2.0. The current release, and the default here. |
| `openfold3` | The earlier OpenFold3 preview-2, Apache-2.0. Kept because earlier results used it. |
| `boltz2` | MIT. Keeps a modified residue as one token; strong on ligands. |
| `protenix2` | Apache-2.0. The widest trunk here (pair channel 256), so the slowest. |
| `rosettafold3` | BSD-3-Clause. Carries chirality features; handles D-amino acids. |
| `chai1` | Apache-2.0. Folds from ESM2 3B, fetched and run automatically. |
| `esmfold2` | MIT. Folds from ESM-C instead of an MSA — single sequence, no search. The 6B tower is a 5.1 GB download. |
| `esmfold2_lm600m` | MIT. Same model against a 600M tower: 0.5 GB instead of 5.1, and no confidence head. |
| `esmfold2_lm300m` | MIT. The smallest tier, 0.3 GB. Also no confidence head. |
| `intellifold2` | Apache-2.0. Widened channels (pair 512), largest download. |
| `opendde` | Apache-2.0. Runs its diffusion on an expanded structural-token set. |
| `af2_ptm` | AlphaFold 2 monomer pTM, CC BY 4.0. **Protein only** — a ligand or nucleotide in the input raises rather than quietly folding the protein part. Templates use the model_1/model_2 parameter sets, the only monomer ones trained with them. |
| `af2_multimer` | AlphaFold 2 multimer v3, CC BY 4.0. Protein only, same as above. |
| `alphafold3` | Google DeepMind's own parameters, under the [AF3 terms of use](https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md). Publicly downloadable now — no login or key — and fetched into `af3_native_weights/`. run_alphafold prints a reminder of the terms at startup. |

Weights for the eleven ported models are downloaded on first use from
[sokrypton/af3-any-model](https://huggingface.co/sokrypton/af3-any-model) into a
per-model cache, so switching models re-downloads only the new one and switching
back is instant.

---

## Download size

| model | download |
|---|---|
| esmfold2_lm300m | 0.12 GB + a 0.3 GB tower |
| esmfold2_lm600m | 0.12 GB + a 0.5 GB tower |
| esmfold2 | 0.17 GB + a 5.1 GB tower |
| protenix2 | 0.18 GB |
| chai1 | 0.25 GB + a 2.4 GB tower |
| openbind0 | 0.25 GB |
| openfold3 | 0.25 GB |
| rosettafold3 | 0.27 GB |
| opendde | 0.33 GB |
| boltz2 | 0.35 GB |
| intellifold2 | 0.59 GB |
| af2_ptm / af2_multimer | 3.5 GB (one tar holds every AlphaFold 2 parameter set) |

`chai1` and the `esmfold2*` models also download a protein language model the
first time they run: 2.4 GB for chai-1, and 5.1 / 0.5 / 0.3 GB for `esmfold2`,
`esmfold2_lm600m` and `esmfold2_lm300m`.

---

## Sequence input

Each molecule type has its own box. Within a box, separate multiple chains with `:`.

| Box | What goes in it | Example |
|---|---|---|
| **protein** | amino-acid sequence(s) | `MKTAY...` or `SEQ1:SEQ2` |
| **dna** | DNA sequence(s) | `CGCGAATTCGCG` |
| **rna** | RNA sequence(s) | `GCGGAUUUA` |
| **ligand_ccd** | ligand(s) by PDB CCD code | `ATP:MG:HEM` |
| **ligand_smiles** | ligand(s) by SMILES | `CC(=O)Oc1ccccc1C(=O)O` |

Chains are assigned IDs A, B, C, … following AlphaFold 3's canonical order (protein → RNA → DNA → ligand; CCD ligands before SMILES ligands). Mix freely across boxes to build a complex — e.g. a protein in **protein**, `AUGCAUGC` in **rna**, and `ATP` in **ligand_ccd**.

- **Homo-oligomers**: identical protein sequences are merged automatically, so `SEQ:SEQ` = homodimer, `SEQ:SEQ:SEQ` = homotrimer.
- Protein / DNA / RNA sequences and CCD codes are upper-cased automatically; **SMILES are left exactly as typed** (case is meaningful in SMILES).
- Spaces and newlines inside an entry are ignored, and **extra colons are forgiven** — `SEQ1::::SEQ2` is the same as `SEQ1:SEQ2`. Leave a box empty if unused.
- *Note:* because `:` separates entries, an atom-mapped SMILES that itself contains a colon (e.g. `[C:1]`) isn't supported via the box — use a raw AF3 JSON for that edge case.

## Seeds

Enter one or more model seeds in the **seeds** box, comma-separated (e.g. `1,2,3`). Each seed is an independent prediction (more seeds = more sampling, more runtime). Non-numeric characters are ignored and duplicates are dropped, so `1, 1, foo, 7` becomes seeds `1` and `7`.

## MSA modes

- **`mmseqs2_server`** *(recommended)*: queries the public [ColabFold](https://colabfold.mmseqs.com/) MMseqs2 API. Covers UniRef30 + environmental sequences for proteins. RNA/DNA chains always run MSA-free (ColabFold is protein-only).
- **`single_sequence`**: no MSA, query sequence only. Faster but less accurate, especially for monomers with close homologs.

## Output files (inside the downloaded zip)

| File | Contents |
|---|---|
| `*.cif` | Best-ranked structure in mmCIF format. B-factor = pLDDT (0–100). |
| `*_confidences.json` | Per-residue pLDDT, PAE matrix, contact probs. |
| `*_summary_confidences.json` | Mean pLDDT, pTM, ipTM, ranking score. |
| `*_ranking_scores.csv` | Ranking scores for all seed × sample combinations. |
| `seed-N_sample-M/` | Individual prediction directories (one per seed/sample). |
| `TERMS_OF_USE.md` | The licence notice for whichever weights you ran. |

## Interpreting confidence scores

- **pLDDT > 90**: very high confidence.
- **pLDDT 70–90**: confident, backbone generally reliable.
- **pLDDT 50–70**: low confidence, treat with caution.
- **pLDDT < 50**: very low, likely disordered or incorrect.
- **PAE**: lower values = confident relative positioning between residue pairs. Useful for assessing interface quality in complexes.
- **ipTM > 0.8**: strong evidence for a well-defined complex interface. **ipTM is `n/a` for single-chain jobs** (there is no interface to score).

## Troubleshooting

- **Check runtime type**: `Runtime → Change runtime type → GPU` (T4 is fine; A100/L4 are faster).
- **OOM error**: reduce sequence length or use a larger-memory GPU runtime.
- **MSA server timeout**: the public ColabFold server is rate-limited. Try again later or switch to `single_sequence` mode.
- **Download popup blocked**: disable your ad blocker.
- **Weight download slow**: the weights are a few hundred MB; the language models for `chai1` and `esmfold2*` are larger.
  The install cell downloads in the background and waits for it automatically.
- **Switching models re-downloads**: each model has its own cache directory,
  so switching back to one you have already used is instant.

## License

The AlphaFold 3 **source code** is [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0).
The **weights** are each their own: Apache-2.0 for openfold3, protenix2, chai1,
intellifold2 and opendde; MIT for boltz2; BSD-3-Clause for rosettafold3. Outputs from
any of those seven are **not** subject to Google DeepMind's AlphaFold 3 Output Terms of
Use and may be used freely, including commercially.

`alphafold3` is the exception: DeepMind's parameters carry their own
[terms of use](https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md),
and the outputs carry DeepMind's output terms. Every run writes a `TERMS_OF_USE.md`
naming the licence that actually applies to it.

## Bugs / feedback

Report issues at https://github.com/sokrypton/colabfold/issues